# Bollinger Bands Breakout Strategy
Buy on lower band touch (mean reversion) vs buy on upper band breakout (trend following).
Which approach works better across different market conditions?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('dark_background')

df = yf.download('SPY', start='2019-01-01', end='2024-12-31', progress=False)
df['sma_20'] = df['Close'].rolling(20).mean()
df['std_20'] = df['Close'].rolling(20).std()
df['bb_upper'] = df['sma_20'] + 2 * df['std_20']
df['bb_lower'] = df['sma_20'] - 2 * df['std_20']
df['bb_pct'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])
df = df.dropna()

In [ ]:
def bb_backtest(df, mode='reversion', capital=100_000):
    """mode: 'reversion' = buy at lower band, 'breakout' = buy above upper band"""
    cash = capital
    shares = 0
    equity = []
    trades = []
    
    for i in range(len(df)):
        price = df['Close'].iloc[i]
        bb_pct = df['bb_pct'].iloc[i]
        
        if mode == 'reversion':
            buy_cond = bb_pct < 0.0  # below lower band
            sell_cond = bb_pct > 0.5  # back to mean
        else:  # breakout
            buy_cond = bb_pct > 1.0  # above upper band
            sell_cond = bb_pct < 0.5  # back to mean
        
        if buy_cond and shares == 0:
            shares = int(cash * 0.95 / price)
            entry = price
            cash -= shares * price
        elif sell_cond and shares > 0:
            pnl = (price - entry) * shares
            trades.append(pnl)
            cash += shares * price
            shares = 0
        
        equity.append(cash + shares * price)
    
    return equity, trades

rev_equity, rev_trades = bb_backtest(df, 'reversion')
brk_equity, brk_trades = bb_backtest(df, 'breakout')

In [ ]:
# Compare
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, rev_equity, label='Mean Reversion', color='#34d399', linewidth=1.2)
ax.plot(df.index, brk_equity, label='Breakout', color='#f97316', linewidth=1.2)
bh = 100_000 * (df['Close'] / df['Close'].iloc[0])
ax.plot(df.index, bh, label='Buy & Hold', color='#71717a', linestyle='--', linewidth=0.8)
ax.set_title('Bollinger Bands: Mean Reversion vs Breakout (SPY)', fontsize=14)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Mean Reversion — Trades: {len(rev_trades)}, Win rate: {sum(1 for t in rev_trades if t > 0)/max(len(rev_trades),1)*100:.0f}%, Final: ${rev_equity[-1]:,.0f}')
print(f'Breakout       — Trades: {len(brk_trades)}, Win rate: {sum(1 for t in brk_trades if t > 0)/max(len(brk_trades),1)*100:.0f}%, Final: ${brk_equity[-1]:,.0f}')